## Real Data Estimation

Virtual patient data (Windkessel model) will be used to test with the method. In this set of data, a total of 10 parameters will be used. The input parameters are: [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

* A previous set of data was used, with only 6 varying parameters. The results can be seen in ../old folder. Some discussions are documented in the latex note.


Import the below functions/libraries vvv

In [1]:
import sys
import os
import pickle

sys.path.insert(0, os.path.abspath(".."))

import numpy as np

from sklearn.preprocessing import StandardScaler

from src.function_library import build_function_library, select_top_power_features, power_features, build_power_library
from src.sparse_interp import sparse_ee_interpretation, save_sparse_result, load_sparse_result, sparse_predict, refine_sparse_result

from npeet import entropy_estimators as ee
import matplotlib.pyplot as plt

#### Pipeline

The function library idea comes from https://doi.org/10.1073/pnas.1517384113. The basic idea is to treat parameter combinations as functions, build a function library, use sparse regression to reduce the dimension of output. A base function library is constructed from the Windkessel parameters using
`build_function_library()`.

The resulting matrix is denoted by:

$$
\Theta_{\mathrm{base}}
$$

Before sparse optimisation, the function library is standardised to avoid scale differences between features.

#### Training

The standardised function library is used to construct a low-dimensional
representation of the Windkessel parameters.

The representation is defined as:

$$
u = \sum_j c_j \widetilde{\Theta}_j(X)
$$

where:

- $\widetilde{\Theta}_j(X)$ is a candidate feature from the standardised
  function library.
- $c_j$ is the coefficient associated with that feature.

The optimisation objective is to maximise the mutual information between
the resulting representation $u$ and the target $y$:

$$
\boxed{
\max_c I(u,y)
}
$$

The optimisation is sparse, so only a subset of the candidate features is
retained in the final representation.

The resulting model is therefore:

$$
\boxed{
u =
\widetilde{\Theta}_{\mathrm{active}}c
}
$$

where `active_indices` identifies the selected features and `coeff`
contains their optimised coefficients.

The result is saved into a `.pkl` file.

### Final data

This dataset contains approximately 800 data points. It was ultimately generated by controlling ventricular pressure based on cleaned_data, after filtering out non-physiological waves.

The experiments involving cleaned_data will be presented in the next section of the notebook.

In [2]:
# loading data
data = np.load(r"D:\Law\25-26\research intern\final_data.npz")
print(data.files)

X_final = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_final = data["v_lv"]
v_rv_final = data["v_rv"]

y_v_lv_max_final = np.max(v_lv_final, axis=1)
y_v_lv_min_final = np.min(v_lv_final, axis=1)

y_v_rv_max_final = np.max(v_rv_final, axis=1)
y_v_rv_min_final = np.min(v_rv_final, axis=1)

y_v_lv_mean_final = np.mean(v_lv_final, axis=1)
y_v_rv_mean_final = np.mean(v_rv_final, axis=1)

['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


Since this dataset is too small, the training set is also used for power feature selection.

In [3]:
# sampling
rng = np.random.default_rng(42)

n_final_samples = len(X_final)

perm_final = rng.permutation(n_final_samples)

n_final_train = int(0.5 * n_final_samples)
n_final_test   = int(0.5 * n_final_samples)

final_train_idx = perm_final[:n_final_train]
final_test_idx   = perm_final[n_final_train:n_final_train + n_final_test]

# --------------------
# Training set
# --------------------
X_final_train = X_final[final_train_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

# --------------------
# Test set
# --------------------
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]
y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

# --------------------
# Save split
# --------------------
final_data_split = {
    "train_idx": final_train_idx,
    "test_idx": final_test_idx
}

with open("final_data_split.pkl", "wb") as f:
    pickle.dump(final_data_split, f)

print("Final data split saved.")

Final data split saved.


In [3]:
# load old sampling
with open("final_data_split.pkl", "rb") as f:
    final_data_split = pickle.load(f)

final_train_idx = final_data_split["train_idx"]
final_test_idx = final_data_split["test_idx"]

X_final_train = X_final[final_train_idx]
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]

y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]

y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]

y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]

y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

### Cleaned Data

This dataset contains only filtered waveforms, however some of them are still considered as non-physiological. This dataset is used as a secondary verification.

In [23]:
# loading data
data = np.load(r"D:\Law\25-26\research intern\cleaned_data.npz")
print(data.files)

X_cleaned = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_cleaned = data["v_lv"]
v_rv_cleaned = data["v_rv"]

y_v_lv_max_cleaned = np.max(v_lv_cleaned, axis=1)
y_v_lv_min_cleaned = np.min(v_lv_cleaned, axis=1)

y_v_rv_max_cleaned = np.max(v_rv_cleaned, axis=1)
y_v_rv_min_cleaned = np.min(v_rv_cleaned, axis=1)

y_v_lv_mean_cleaned = np.mean(v_lv_cleaned, axis=1)
y_v_rv_mean_cleaned = np.mean(v_rv_cleaned, axis=1)

['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


In [24]:
# sampling
rng = np.random.default_rng(42)

n_cleaned_samples = len(X_cleaned)

perm_cleaned = rng.permutation(n_cleaned_samples)

n_cleaned_train = int(0.5 * n_cleaned_samples)
n_cleaned_test  = int(0.5 * n_cleaned_samples)

cleaned_train_idx = perm_cleaned[:n_cleaned_train]
cleaned_test_idx  = perm_cleaned[
    n_cleaned_train:n_cleaned_train + n_cleaned_test
]

# --------------------
# Training set
# --------------------
X_cleaned_train = X_cleaned[cleaned_train_idx]

y_v_lv_max_cleaned_train = y_v_lv_max_cleaned[cleaned_train_idx]
y_v_lv_min_cleaned_train = y_v_lv_min_cleaned[cleaned_train_idx]
y_v_rv_max_cleaned_train = y_v_rv_max_cleaned[cleaned_train_idx]
y_v_rv_min_cleaned_train = y_v_rv_min_cleaned[cleaned_train_idx]
y_v_lv_mean_cleaned_train = y_v_lv_mean_cleaned[cleaned_train_idx]
y_v_rv_mean_cleaned_train = y_v_rv_mean_cleaned[cleaned_train_idx]

# --------------------
# Test set
# --------------------
X_cleaned_test = X_cleaned[cleaned_test_idx]

y_v_lv_max_cleaned_test = y_v_lv_max_cleaned[cleaned_test_idx]
y_v_lv_min_cleaned_test = y_v_lv_min_cleaned[cleaned_test_idx]
y_v_rv_max_cleaned_test = y_v_rv_max_cleaned[cleaned_test_idx]
y_v_rv_min_cleaned_test = y_v_rv_min_cleaned[cleaned_test_idx]
y_v_lv_mean_cleaned_test = y_v_lv_mean_cleaned[cleaned_test_idx]
y_v_rv_mean_cleaned_test = y_v_rv_mean_cleaned[cleaned_test_idx]

# --------------------
# Save split
# --------------------
cleaned_data_split = {
    "train_idx": cleaned_train_idx,
    "test_idx": cleaned_test_idx
}

with open("cleaned_data_split.pkl", "wb") as f:
    pickle.dump(cleaned_data_split, f)

print("Cleaned data split saved.")

Cleaned data split saved.


In [25]:
# load old sampling
with open("cleaned_data_split.pkl", "rb") as f:
    cleaned_data_split = pickle.load(f)

cleaned_train_idx = cleaned_data_split["train_idx"]
cleaned_test_idx = cleaned_data_split["test_idx"]

X_cleaned_train = X_cleaned[cleaned_train_idx]
X_cleaned_test = X_cleaned[cleaned_test_idx]

y_v_lv_max_cleaned_train = y_v_lv_max_cleaned[cleaned_train_idx]
y_v_lv_max_cleaned_test = y_v_lv_max_cleaned[cleaned_test_idx]

y_v_lv_min_cleaned_train = y_v_lv_min_cleaned[cleaned_train_idx]
y_v_lv_min_cleaned_test = y_v_lv_min_cleaned[cleaned_test_idx]

y_v_rv_max_cleaned_train = y_v_rv_max_cleaned[cleaned_train_idx]
y_v_rv_max_cleaned_test = y_v_rv_max_cleaned[cleaned_test_idx]

y_v_rv_min_cleaned_train = y_v_rv_min_cleaned[cleaned_train_idx]
y_v_rv_min_cleaned_test = y_v_rv_min_cleaned[cleaned_test_idx]

y_v_lv_mean_cleaned_train = y_v_lv_mean_cleaned[cleaned_train_idx]
y_v_lv_mean_cleaned_test = y_v_lv_mean_cleaned[cleaned_test_idx]

y_v_rv_mean_cleaned_train = y_v_rv_mean_cleaned[cleaned_train_idx]
y_v_rv_mean_cleaned_test = y_v_rv_mean_cleaned[cleaned_test_idx]

### Training

In [5]:
Theta_v_lv_max_final, \
feature_names_v_lv_max_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_lv_max_final = StandardScaler()

Theta_scaled_v_lv_max_final = (
    scaler_v_lv_max_final.fit_transform(
        Theta_v_lv_max_final
    )
)

sparse_result_v_lv_max_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_max_final,
    y_v_lv_max_final_train,
    feature_names_v_lv_max_final,
    scaler_v_lv_max_final,
    threshold=0.05,
    resume=True
)

refined_result_v_lv_max_final = refine_sparse_result(
    sparse_result_v_lv_max_final,
    Theta_scaled_v_lv_max_final,
    y_v_lv_max_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_lv_max_final,
    filename="../results/sparse_result_v_lv_max_final.pkl"
)


===== Iteration 1 =====
MI              : 1.889313
Active features : 75
C_p                                :  2.288869  KEEP
Za_p                               : -2.677541  KEEP
R_p                                :  0.480470  KEEP
Emax_rv                            : -6.095592  KEEP
Emin_rv                            : -0.211767  KEEP
C_s                                :  1.555196  KEEP
Za_s                               :  0.173127  KEEP
R_s                                :  0.266558  KEEP
Emax_lv                            :  8.355386  KEEP
Emin_lv                            :  0.392020  KEEP
C_p^2                              :  0.321021  KEEP
Za_p^2                             :  0.162327  KEEP
R_p^2                              :  2.252863  KEEP
Emax_rv^2                          :  0.161981  KEEP
Emin_rv^2                          :  0.724513  KEEP
C_s^2                              :  1.562166  KEEP
Za_s^2                             :  0.028536  REMOVE
R_s^2                   


===== Leave-one-out Feature Importance =====
Full MI : 1.933170

C_p                                : coeff=-1.545675  MI(-f)= 1.927194  ΔMI= 0.005976  KEEP
Za_p                               : coeff= 2.867832  MI(-f)= 1.781249  ΔMI= 0.151921  KEEP
R_p                                : coeff=-1.187783  MI(-f)= 1.946093  ΔMI=-0.012923  REMOVE
Emax_rv                            : coeff= 6.455900  MI(-f)= 1.241697  ΔMI= 0.691474  KEEP
Emin_rv                            : coeff= 1.106920  MI(-f)= 2.013712  ΔMI=-0.080541  REMOVE
C_s                                : coeff=-1.085810  MI(-f)= 1.955741  ΔMI=-0.022571  REMOVE
Za_s                               : coeff=-0.676333  MI(-f)= 1.930749  ΔMI= 0.002421  KEEP
R_s                                : coeff=-0.647866  MI(-f)= 1.958602  ΔMI=-0.025432  REMOVE
Emax_lv                            : coeff=-6.161643  MI(-f)= 1.528386  ΔMI= 0.404785  KEEP
Emin_lv                            : coeff=-0.314879  MI(-f)= 1.957759  ΔMI=-0.024589  REMOVE
C_p^

In [7]:
Theta_v_lv_min_final, \
feature_names_v_lv_min_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_lv_min_final = StandardScaler()

Theta_scaled_v_lv_min_final = (
    scaler_v_lv_min_final.fit_transform(
        Theta_v_lv_min_final
    )
)

sparse_result_v_lv_min_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_min_final,
    y_v_lv_min_final_train,
    feature_names_v_lv_min_final,
    scaler_v_lv_min_final,
    threshold=0.1,
    resume=True
)

refined_result_v_lv_min_final = refine_sparse_result(
    sparse_result_v_lv_min_final,
    Theta_scaled_v_lv_min_final,
    y_v_lv_min_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_lv_min_final,
    filename="../results/sparse_result_v_lv_min_final.pkl"
)


===== Iteration 1 =====
MI              : 0.321116
Active features : 75
C_p                                : -0.217709  KEEP
Za_p                               :  0.214620  KEEP
R_p                                : -0.125382  KEEP
Emax_rv                            :  0.192556  KEEP
Emin_rv                            :  0.077438  REMOVE
C_s                                :  0.200620  KEEP
Za_s                               :  0.155178  KEEP
R_s                                :  0.094677  REMOVE
Emax_lv                            :  0.202522  KEEP
Emin_lv                            :  0.200128  KEEP
C_p^2                              :  0.170380  KEEP
Za_p^2                             :  0.237947  KEEP
R_p^2                              :  0.130677  KEEP
Emax_rv^2                          :  0.199543  KEEP
Emin_rv^2                          :  0.202882  KEEP
C_s^2                              :  0.241299  KEEP
Za_s^2                             :  0.206221  KEEP
R_s^2                 


===== Iteration 4 =====
MI              : 1.254995
Active features : 2
Emax_lv                            :  13.767152  KEEP
Za_p*R_p                           : -0.920951  KEEP

===== Final Sparse Combination =====
Maximum MI : 1.252930
Terms      : 2
Emax_lv                            : 13.767573
Za_p*R_p                           : -0.961282

u ∝
13.7676*Emax_lv + -0.9613*Za_p*R_p

===== Coefficient Pre-screening =====
Original features : 2
Remaining features: 2

Emax_lv                            :  13.767573  KEEP
Za_p*R_p                           : -0.961282  KEEP

===== Leave-one-out Feature Importance =====
Full MI : 1.254689

Emax_lv                            : coeff= 13.767104  MI(-f)=-0.007069  ΔMI= 1.261758  KEEP
Za_p*R_p                           : coeff=-0.917424  MI(-f)= 1.495551  ΔMI=-0.240862  REMOVE

===== Refined Sparse Combination =====
Initial terms : 2
Final terms   : 1
Final MI      : 1.161660

Emax_lv                            : 13.693246

u ∝
13.6932*Emax_l

In [8]:
Theta_v_rv_max_final, \
feature_names_v_rv_max_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_rv_max_final = StandardScaler()

Theta_scaled_v_rv_max_final = (
    scaler_v_rv_max_final.fit_transform(
        Theta_v_rv_max_final
    )
)

sparse_result_v_rv_max_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_max_final,
    y_v_rv_max_final_train,
    feature_names_v_rv_max_final,
    scaler_v_rv_max_final,
    threshold=0.1,
    resume=True
)

refined_result_v_rv_max_final = refine_sparse_result(
    sparse_result_v_rv_max_final,
    Theta_scaled_v_rv_max_final,
    y_v_rv_max_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_rv_max_final,
    filename="../results/sparse_result_v_rv_max_final.pkl"
)


===== Iteration 1 =====
MI              : 1.946094
Active features : 75
C_p                                : -4.968072  KEEP
Za_p                               : -0.310720  KEEP
R_p                                : -1.089740  KEEP
Emax_rv                            : -0.675126  KEEP
Emin_rv                            : -0.854407  KEEP
C_s                                : -0.303673  KEEP
Za_s                               : -0.839792  KEEP
R_s                                : -2.394448  KEEP
Emax_lv                            :  1.267595  KEEP
Emin_lv                            : -0.412300  KEEP
C_p^2                              : -0.305568  KEEP
Za_p^2                             : -0.279978  KEEP
R_p^2                              : -0.208709  KEEP
Emax_rv^2                          : -1.006437  KEEP
Emin_rv^2                          : -0.849920  KEEP
C_s^2                              :  1.315698  KEEP
Za_s^2                             : -1.185471  KEEP
R_s^2                     


===== Leave-one-out Feature Importance =====
Full MI : 1.946094

C_p                                : coeff=-4.968072  MI(-f)= 1.651414  ΔMI= 0.294680  KEEP
Za_p                               : coeff=-0.310720  MI(-f)= 2.084591  ΔMI=-0.138497  REMOVE
R_p                                : coeff=-1.089740  MI(-f)= 2.067393  ΔMI=-0.121299  REMOVE
Emax_rv                            : coeff=-0.675126  MI(-f)= 2.051100  ΔMI=-0.105006  REMOVE
Emin_rv                            : coeff=-0.854407  MI(-f)= 2.027011  ΔMI=-0.080917  REMOVE
C_s                                : coeff=-0.303673  MI(-f)= 2.001604  ΔMI=-0.055510  REMOVE
Za_s                               : coeff=-0.839792  MI(-f)= 2.065766  ΔMI=-0.119673  REMOVE
R_s                                : coeff=-2.394448  MI(-f)= 2.024849  ΔMI=-0.078755  REMOVE
Emax_lv                            : coeff= 1.267595  MI(-f)= 2.006190  ΔMI=-0.060096  REMOVE
Emin_lv                            : coeff=-0.412300  MI(-f)= 2.077122  ΔMI=-0.131028  REM

In [9]:
Theta_v_rv_min_final, \
feature_names_v_rv_min_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_rv_min_final = StandardScaler()

Theta_scaled_v_rv_min_final = (
    scaler_v_rv_min_final.fit_transform(
        Theta_v_rv_min_final
    )
)

sparse_result_v_rv_min_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_min_final,
    y_v_rv_min_final_train,
    feature_names_v_rv_min_final,
    scaler_v_rv_min_final,
    threshold=0.1,
    resume=True
)

refined_result_v_rv_min_final = refine_sparse_result(
    sparse_result_v_rv_min_final,
    Theta_scaled_v_rv_min_final,
    y_v_rv_min_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_rv_min_final,
    filename="../results/sparse_result_v_rv_min_final.pkl"
)


===== Iteration 1 =====
MI              : 0.541674
Active features : 75
C_p                                :  1.897586  KEEP
Za_p                               :  0.374987  KEEP
R_p                                :  0.893710  KEEP
Emax_rv                            :  0.412629  KEEP
Emin_rv                            :  0.373493  KEEP
C_s                                :  0.372841  KEEP
Za_s                               :  0.372522  KEEP
R_s                                :  0.339611  KEEP
Emax_lv                            :  0.128440  KEEP
Emin_lv                            :  0.121852  KEEP
C_p^2                              :  0.549093  KEEP
Za_p^2                             :  0.380904  KEEP
R_p^2                              :  0.321761  KEEP
Emax_rv^2                          :  0.373517  KEEP
Emin_rv^2                          :  0.373862  KEEP
C_s^2                              :  1.316297  KEEP
Za_s^2                             :  0.316975  KEEP
R_s^2                     


===== Final Sparse Combination =====
Maximum MI : 1.425075
Terms      : 17
C_p                                : 3.173348
Emax_rv                            : 13.532321
Za_s                               : -1.469516
R_s                                : 1.152395
Emax_lv                            : -1.725440
Emax_rv^2                          : 2.795040
Emin_rv^2                          : 1.184625
C_s^2                              : 0.663173
log(R_s)                           : 0.667924
C_p*C_s                            : 1.386401
C_p*Za_s                           : 1.422922
Za_p*Emax_rv                       : 1.542616
Za_p*Emin_rv                       : 0.583152
Za_p*C_s                           : 0.623290
C_s*Emin_lv                        : 0.673800
Za_s*R_s                           : 0.656035
Za_s*Emax_lv                       : 0.628686

u ∝
3.1733*C_p + 13.5323*Emax_rv + -1.4695*Za_s + 1.1524*R_s + -1.7254*Emax_lv + 2.7950*Emax_rv^2 + 1.1846*Emin_rv^2 + 0.6632*C_s^2 + 0.66

In [10]:
Theta_v_lv_mean_final, \
feature_names_v_lv_mean_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_lv_mean_final = StandardScaler()

Theta_scaled_v_lv_mean_final = (
    scaler_v_lv_mean_final.fit_transform(
        Theta_v_lv_mean_final
    )
)

sparse_result_v_lv_mean_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_mean_final,
    y_v_lv_mean_final_train,
    feature_names_v_lv_mean_final,
    scaler_v_lv_mean_final,
    threshold=0.1,
    resume=True
)

refined_result_v_lv_mean_final = refine_sparse_result(
    sparse_result_v_lv_mean_final,
    Theta_scaled_v_lv_mean_final,
    y_v_lv_mean_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_lv_mean_final,
    filename="../results/sparse_result_v_lv_mean_final.pkl"
)


===== Iteration 1 =====
MI              : 1.862178
Active features : 75
C_p                                :  0.890297  KEEP
Za_p                               :  0.150256  KEEP
R_p                                :  0.108700  KEEP
Emax_rv                            :  0.104258  KEEP
Emin_rv                            :  0.110801  KEEP
C_s                                :  0.107915  KEEP
Za_s                               :  0.110610  KEEP
R_s                                :  0.091208  REMOVE
Emax_lv                            :  0.140113  KEEP
Emin_lv                            :  0.112926  KEEP
C_p^2                              :  0.109423  KEEP
Za_p^2                             :  0.110764  KEEP
R_p^2                              :  0.113421  KEEP
Emax_rv^2                          :  0.044401  REMOVE
Emin_rv^2                          :  0.264660  KEEP
C_s^2                              :  0.116872  KEEP
Za_s^2                             :  0.019413  REMOVE
R_s^2               


===== Iteration 4 =====
MI              : 1.950451
Active features : 56
C_p                                :  0.614783  KEEP
Za_p                               : -1.118220  KEEP
R_p                                :  0.170463  KEEP
Emax_rv                            : -1.758268  KEEP
Emin_rv                            :  0.228709  KEEP
C_s                                :  0.197761  KEEP
Za_s                               :  0.194058  KEEP
Emax_lv                            :  9.837993  KEEP
C_p^2                              : -0.198052  KEEP
Za_p^2                             :  0.258589  KEEP
R_s^2                              : -0.292164  KEEP
Emin_lv^2                          :  0.283914  KEEP
log(C_p)                           :  0.178265  KEEP
log(R_p)                           :  0.175188  KEEP
log(C_s)                           :  0.775563  KEEP
log(Emax_lv)                       :  1.860141  KEEP
log(Emin_lv)                       :  0.172747  KEEP
C_p*Za_p                  


===== Iteration 7 =====
MI              : 1.889857
Active features : 49
C_p                                :  0.220756  KEEP
Za_p                               :  0.162513  KEEP
Emax_rv                            : -0.634131  KEEP
Emin_rv                            :  0.167482  KEEP
C_s                                :  0.167909  KEEP
Emax_lv                            :  10.589788  KEEP
C_p^2                              :  0.167421  KEEP
Za_p^2                             : -0.873992  KEEP
Emin_lv^2                          :  0.172988  KEEP
log(C_p)                           :  0.130151  KEEP
log(R_p)                           :  0.613674  KEEP
log(C_s)                           :  0.390788  KEEP
log(Emax_lv)                       :  0.804356  KEEP
log(Emin_lv)                       :  0.239392  KEEP
C_p*Za_p                           :  0.173494  KEEP
C_p*R_p                            :  1.390723  KEEP
C_p*Emax_rv                        :  0.168376  KEEP
C_p*Emin_rv              


===== Leave-one-out Feature Importance =====
Full MI : 1.889857

C_p                                : coeff= 0.220756  MI(-f)= 2.022179  ΔMI=-0.132321  REMOVE
Za_p                               : coeff= 0.162513  MI(-f)= 2.045918  ΔMI=-0.156061  REMOVE
Emax_rv                            : coeff=-0.634131  MI(-f)= 1.900234  ΔMI=-0.010376  REMOVE
Emin_rv                            : coeff= 0.167482  MI(-f)= 2.010048  ΔMI=-0.120191  REMOVE
C_s                                : coeff= 0.167909  MI(-f)= 2.015029  ΔMI=-0.125172  REMOVE
Emax_lv                            : coeff= 10.589788  MI(-f)= 0.842139  ΔMI= 1.047719  KEEP
C_p^2                              : coeff= 0.167421  MI(-f)= 2.029391  ΔMI=-0.139534  REMOVE
Za_p^2                             : coeff=-0.873992  MI(-f)= 1.936529  ΔMI=-0.046672  REMOVE
Emin_lv^2                          : coeff= 0.172988  MI(-f)= 1.979791  ΔMI=-0.089934  REMOVE
log(C_p)                           : coeff= 0.130151  MI(-f)= 2.025360  ΔMI=-0.135503  RE

In [11]:
Theta_v_rv_mean_final, \
feature_names_v_rv_mean_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_rv_mean_final = StandardScaler()

Theta_scaled_v_rv_mean_final = (
    scaler_v_rv_mean_final.fit_transform(
        Theta_v_rv_mean_final
    )
)

sparse_result_v_rv_mean_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_mean_final,
    y_v_rv_mean_final_train,
    feature_names_v_rv_mean_final,
    scaler_v_rv_mean_final,
    threshold=0.1,
    resume=True
)

refined_result_v_rv_mean_final = refine_sparse_result(
    sparse_result_v_rv_mean_final,
    Theta_scaled_v_rv_mean_final,
    y_v_rv_mean_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_rv_mean_final,
    filename="../results/sparse_result_v_rv_mean_final.pkl"
)


===== Iteration 1 =====
MI              : 1.249304
Active features : 75
C_p                                :  0.061285  REMOVE
Za_p                               :  0.099460  REMOVE
R_p                                :  0.099359  REMOVE
Emax_rv                            :  13.221489  KEEP
Emin_rv                            :  0.115645  KEEP
C_s                                : -0.074965  REMOVE
Za_s                               : -1.008952  KEEP
R_s                                :  0.099170  REMOVE
Emax_lv                            :  0.201562  KEEP
Emin_lv                            :  0.100199  KEEP
C_p^2                              :  2.942578  KEEP
Za_p^2                             : -0.350753  KEEP
R_p^2                              : -1.251696  KEEP
Emax_rv^2                          :  2.983409  KEEP
Emin_rv^2                          :  0.395334  KEEP
C_s^2                              :  0.338660  KEEP
Za_s^2                             :  0.101704  KEEP
R_s^2          


===== Leave-one-out Feature Importance =====
Full MI : 1.805774

Emax_rv                            : coeff= 16.102293  MI(-f)= 0.789828  ΔMI= 1.015946  KEEP
Emin_rv                            : coeff= 0.476207  MI(-f)= 1.974070  ΔMI=-0.168297  REMOVE
Emax_lv                            : coeff=-0.530208  MI(-f)= 1.971157  ΔMI=-0.165384  REMOVE
Emin_lv                            : coeff=-0.645452  MI(-f)= 1.957262  ΔMI=-0.151489  REMOVE
C_p^2                              : coeff= 4.546119  MI(-f)= 1.572106  ΔMI= 0.233667  KEEP
Za_p^2                             : coeff=-0.752075  MI(-f)= 1.902183  ΔMI=-0.096409  REMOVE
R_p^2                              : coeff=-4.316349  MI(-f)= 1.649274  ΔMI= 0.156500  KEEP
Emax_rv^2                          : coeff=-8.008621  MI(-f)= 1.707573  ΔMI= 0.098201  KEEP
Emin_rv^2                          : coeff= 0.715176  MI(-f)= 1.957689  ΔMI=-0.151915  REMOVE
C_s^2                              : coeff= 0.314547  MI(-f)= 2.024776  ΔMI=-0.219003  REMOVE
Z

In [26]:
Theta_v_lv_max_cleaned, \
feature_names_v_lv_max_cleaned = build_function_library(
    X_cleaned_train,
    param_names
)

scaler_v_lv_max_cleaned = StandardScaler()

Theta_scaled_v_lv_max_cleaned = (
    scaler_v_lv_max_cleaned.fit_transform(
        Theta_v_lv_max_cleaned
    )
)

sparse_result_v_rv_mean_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_mean_final,
    y_v_rv_mean_final_train,
    feature_names_v_rv_mean_final,
    scaler_v_rv_mean_final,
    threshold=0.1,
    resume=True
)

save_sparse_result(
    sparse_result_v_lv_max_cleaned,
    filename="../results/sparse_result_v_lv_max_cleaned.pkl"
)


===== Iteration 1 =====
MI              : 2.543355
Active features : 75
C_p                                :  1.728151  KEEP
Za_p                               : -2.248532  KEEP
R_p                                :  9.969474  KEEP
Emax_rv                            : -0.064222  REMOVE
Emin_rv                            : -0.541987  KEEP
C_s                                :  5.312689  KEEP
Za_s                               : -2.505985  KEEP
R_s                                : -2.971618  KEEP
Emax_lv                            :  9.993041  KEEP
Emin_lv                            :  0.246095  KEEP
C_p^2                              : -0.479678  KEEP
Za_p^2                             :  0.265985  KEEP
R_p^2                              :  0.144622  KEEP
Emax_rv^2                          : -1.301949  KEEP
Emin_rv^2                          :  0.181725  KEEP
C_s^2                              : -0.830412  KEEP
Za_s^2                             : -2.201206  KEEP
R_s^2                   


===== Iteration 4 =====
MI              : 1.725376
Active features : 58
C_p                                :  0.645592  KEEP
Za_p                               : -1.950901  KEEP
R_p                                :  5.582530  KEEP
Emin_rv                            : -0.112016  KEEP
C_s                                :  3.866660  KEEP
Za_s                               : -2.097868  KEEP
R_s                                : -2.144100  KEEP
Emax_lv                            :  6.193194  KEEP
C_p^2                              :  0.470528  KEEP
Za_p^2                             :  0.156146  KEEP
R_p^2                              :  1.231767  KEEP
Emax_rv^2                          : -2.210375  KEEP
Emin_rv^2                          : -1.088491  KEEP
C_s^2                              :  0.538016  KEEP
Za_s^2                             : -1.167362  KEEP
R_s^2                              :  0.459413  KEEP
Emax_lv^2                          :  0.354748  KEEP
log(C_p)                  


===== Iteration 7 =====
MI              : 1.746734
Active features : 48
C_p                                :  0.551538  KEEP
R_p                                :  5.904195  KEEP
Emin_rv                            : -1.544652  KEEP
C_s                                :  1.955119  KEEP
Za_s                               : -2.178766  KEEP
R_s                                : -0.473206  KEEP
Emax_lv                            :  6.041467  KEEP
C_p^2                              :  0.688882  KEEP
Za_p^2                             : -1.219288  KEEP
R_p^2                              :  0.339526  KEEP
Emax_rv^2                          : -2.800117  KEEP
Emin_rv^2                          :  0.442839  KEEP
C_s^2                              :  1.674442  KEEP
Za_s^2                             : -0.432188  KEEP
R_s^2                              : -0.829301  KEEP
log(C_p)                           :  1.664736  KEEP
log(R_p)                           :  0.679934  KEEP
log(Emax_rv)              


===== Final Sparse Combination =====
Maximum MI : 1.778421
Terms      : 39
R_p                                : 5.514876
Emin_rv                            : -0.891190
C_s                                : 2.893015
Za_s                               : -2.199492
R_s                                : -1.322501
Emax_lv                            : 6.356778
C_p^2                              : 1.340316
Za_p^2                             : -0.663436
R_p^2                              : 0.885218
Emax_rv^2                          : -2.190002
Emin_rv^2                          : -0.206880
C_s^2                              : 0.694694
Za_s^2                             : -1.000232
log(C_p)                           : 1.298707
log(Emax_rv)                       : -0.615176
log(C_s)                           : 1.590784
log(Emax_lv)                       : 1.422426
C_p*R_p                            : 2.441636
C_p*Emax_rv                        : -0.543829
C_p*Emin_lv                        : 0.35